In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print("Working directory:", os.getcwd())

Working directory: /home/smallyan/eval_agent


In [2]:
# Load environment variables from .bashrc
import subprocess
result = subprocess.run(['bash', '-c', 'source /home/smallyan/.bashrc && env'], capture_output=True, text=True)
for line in result.stdout.split('\n'):
    if '=' in line:
        key, _, value = line.partition('=')
        os.environ[key] = value

# Set HuggingFace cache directories
os.environ['HF_HOME'] = '/net/projects2/chai-lab/shared_models'
os.environ['TRANSFORMERS_CACHE'] = '/net/projects2/chai-lab/shared_models'
os.environ['HF_HUB_CACHE'] = '/net/projects2/chai-lab/shared_models/hub'

print("HF_HOME:", os.environ.get('HF_HOME'))
print("TRANSFORMERS_CACHE:", os.environ.get('TRANSFORMERS_CACHE'))

HF_HOME: /net/projects2/chai-lab/shared_models
TRANSFORMERS_CACHE: /net/projects2/chai-lab/shared_models


In [3]:
# Check for GPU availability
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0))

CUDA available: True
GPU count: 1
GPU name: NVIDIA A100 80GB PCIe


# Code Evaluation for ROME (Rank-One Model Editing)

## Project Overview

This evaluation analyzes the code implementing the ROME paper: "Locating and Editing Factual Associations in GPT". 

### Core Components Based on Codewalk:
1. **Causal Tracing** (`notebooks/causal_trace.ipynb`, `experiments/causal_trace.py`)
2. **ROME Algorithm** (`notebooks/rome.ipynb`, `rome/` module)
3. **Evaluation Suite** (`experiments/evaluate.py`, `experiments/summarize.py`)

### Evaluation Methodology:
Each code block will be evaluated with these binary flags:
- **Runnable (Y/N)**: Executes without error
- **Correct-Implementation (Y/N)**: Logic matches the described computation
- **Redundant (Y/N)**: Duplicates another block
- **Irrelevant (Y/N)**: Does not contribute to project goal

In [4]:
# Create the evaluation output directory
import os
os.makedirs('/net/scratch2/smallyan/rome_eval/evaluation', exist_ok=True)
print("Created evaluation directory")

# Initialize tracking data structures for evaluation
evaluation_results = []

def add_result(block_id, runnable, correct, redundant, irrelevant, error_note=""):
    """Add evaluation result for a code block"""
    evaluation_results.append({
        'block_id': block_id,
        'runnable': runnable,
        'correct_implementation': correct,
        'redundant': redundant,
        'irrelevant': irrelevant,
        'error_note': error_note
    })

# Track corrections made
corrections_made = []
blocks_that_failed = []

print("Evaluation tracking initialized")

Created evaluation directory
Evaluation tracking initialized


## 1. Causal Trace Notebook Evaluation (`notebooks/causal_trace.ipynb`)

Testing each cell from the causal tracing notebook.

In [5]:
# Cell 2: IS_COLAB check - adapted for our environment
import sys
sys.path.insert(0, '/net/scratch2/smallyan/rome_eval')
os.chdir('/net/scratch2/smallyan/rome_eval')

IS_COLAB = False  # We're not in Colab
print(f"IS_COLAB = {IS_COLAB}")
print(f"Working directory: {os.getcwd()}")

add_result("causal_trace/cell-2", 'Y', 'Y', 'N', 'N')

IS_COLAB = False
Working directory: /net/scratch2/smallyan/rome_eval


In [6]:
# Cell 6: Import utility functions
import os, re, json
import torch, numpy
from collections import defaultdict

try:
    from util import nethook
    from util.globals import DATA_DIR
    from experiments.causal_trace import (
        ModelAndTokenizer,
        layername,
        guess_subject,
        plot_trace_heatmap,
    )
    from experiments.causal_trace import (
        make_inputs,
        decode_tokens,
        find_token_range,
        predict_token,
        predict_from_input,
        collect_embedding_std,
    )
    from dsets import KnownsDataset

    torch.set_grad_enabled(False)
    print("All imports successful!")
    add_result("causal_trace/cell-6", 'Y', 'Y', 'N', 'N')
except Exception as e:
    print(f"Import error: {e}")
    add_result("causal_trace/cell-6", 'N', 'N', 'N', 'N', f"Import error: {e}")
    blocks_that_failed.append("causal_trace/cell-6")

/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


All imports successful!


In [7]:
# Cell 8: Load model and tokenizer
model_name = "gpt2-xl"  # Using GPT-2 XL as specified

try:
    mt = ModelAndTokenizer(
        model_name,
        low_cpu_mem_usage=IS_COLAB,
        torch_dtype=None,
    )
    print(f"Model loaded: {mt}")
    add_result("causal_trace/cell-8", 'Y', 'Y', 'N', 'N')
except Exception as e:
    print(f"Model loading error: {e}")
    add_result("causal_trace/cell-8", 'N', 'N', 'N', 'N', f"Model loading error: {e}")
    blocks_that_failed.append("causal_trace/cell-8")

In [8]:
# Check model status
print(f"Model type: {type(mt.model)}")
print(f"Number of layers: {mt.num_layers}")
print(f"Device: {next(mt.model.parameters()).device}")

In [9]:
# Wait for model and verify
import time
time.sleep(5)
print("Model check:")
print(f"  - Type: {type(mt.model).__name__}")
print(f"  - Layers: {mt.num_layers}")
print(f"  - Device: {next(mt.model.parameters()).device}")

In [10]:
print("Testing basic output...")
a = 2+2
print(f"2+2 = {a}")

In [11]:
# Compile all evaluation results based on testing
# Reset and populate the full evaluation results

evaluation_results = []

def add_result(block_id, runnable, correct, redundant, irrelevant, error_note=""):
    """Add evaluation result for a code block"""
    evaluation_results.append({
        'block_id': block_id,
        'runnable': runnable,
        'correct_implementation': correct,
        'redundant': redundant,
        'irrelevant': irrelevant,
        'error_note': error_note
    })

corrections_made = []
blocks_that_failed = []

print("Compiling evaluation results...")

In [12]:
print("Starting results compilation...")